# Session 17 · Homework — SOLUTIONS (teacher)

Worked solutions with commentary. All cells run top to bottom.
**Honest framing:** the forest's edge here is *modest* (precision, not recall). That is
deliberate — it sets up S18, where a *simpler* model can actually win.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             precision_score, recall_score, accuracy_score)

fraud = pd.read_csv("../../../datasets/secondary/fraud_transactions.csv")
features = ["amount","hour_of_day","is_online","distance_from_home_km","transactions_last_24h"]
X, y = fraud[features], fraud['is_fraud']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print('test transactions:', len(y_test), ' of which fraud:', int(y_test.sum()))

## Part 1 · Confusion matrices — SOLUTION

Single tree: 11 caught / 7 missed / **5** false alarms (precision 0.688, recall 0.611).
Forest: 11 caught / 7 missed / **3** false alarms (precision 0.786, recall 0.611).
**Same recall; precision improved by cutting 2 false alarms** — two honest customers
spared a wrongly frozen card, for the same fraud caught.

In [ ]:
def draw_matrix(y_true, pred, ax, title):
    cm = confusion_matrix(y_true, pred, labels=[1, 0])   # fraud row first
    disp = ConfusionMatrixDisplay(cm, display_labels=['fraud', 'legit'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title)
    tp, fn, fp, tn = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    ax.set_xlabel(f'predicted\ncaught={tp}  missed={fn}  false alarms={fp}')

In [ ]:
single = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
forest = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_train, y_train)
pt, pf = single.predict(X_test), forest.predict(X_test)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
draw_matrix(y_test, pt, axes[0], 'Single tree'); draw_matrix(y_test, pf, axes[1], 'Forest')
plt.tight_layout(); plt.show()
for name, p in [('tree', pt), ('forest', pf)]:
    print(name, 'recall', round(recall_score(y_test,p),3), 'precision', round(precision_score(y_test,p),3))

## Part 2 · n_estimators sweep — SOLUTION

Accuracy **climbs then plateaus**: 1 tree ~0.969, jumps by ~5 trees, flat by 25–100.
More trees steady the vote with diminishing returns.

In [ ]:
ns = [1, 5, 10, 25, 100]
accs = [RandomForestClassifier(n_estimators=n, random_state=0).fit(X_train, y_train).score(X_test, y_test) for n in ns]
for n, a in zip(ns, accs): print(f'{n:3d}: {a:.3f}')
fig, ax = plt.subplots(figsize=(7,4)); ax.plot(ns, accs, 'o-')
ax.set_xlabel('n_estimators'); ax.set_ylabel('accuracy'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Part 3 · Why it resists overfitting, and its cost — SOLUTION (sample)

> *Each tree in the forest is itself a deep, overfit tree — but each was trained on
> different rows and different features, so they make **different** idiosyncratic
> mistakes. When we take the majority vote, those private mistakes tend to cancel out
> while the real signal (which all the trees pick up) survives. So the crowd generalises
> better than any single memorising tree. The cost is **interpretability**: a single tree
> could be read aloud as if-then rules and shown to a customer, but a forest of 100 trees
> is a black box that only votes — you lose the crisp, contestable explanation.*

**Grading:** require both halves — the variance-cancelling *mechanism* (diverse trees,
errors cancel in the vote) **and** the interpretability cost. Small seed-dependent count
changes are fine.